# 注意

- 簡便化のため、プロトコルを大胆に改変しています
- 簡便化のため、一部脆弱な実装を含みます
- データフォーマットは独自のものを使っています
- 間違いが含まれてる可能性があります

# 0. 前準備

## 0.1 ルートCA証明書のインポート

In [ ]:
# @title  {"run":"auto"}
ca_crt = "" # @param {"type":"string","placeholder":"CA証明書の中身を貼ってください"}

# crt形式に成形
ca_crt = ca_crt.replace(" ", "\n")
ca_crt = ca_crt.replace("-----BEGIN\nCERTIFICATE-----", "-----BEGIN CERTIFICATE-----")
ca_crt = ca_crt.replace("-----END\nCERTIFICATE-----", "-----END CERTIFICATE-----")
open("ca.crt", "w").write(ca_crt)
print(ca_crt)
print("アップロード完了")


アップロード完了


## 0.2 自分の鍵を作ろう

In [ ]:
# サーバー秘密鍵を生成
!openssl genrsa -out server.key 2048
# CSR（証明書署名要求）を作成
!openssl req -new \
  -key server.key \
  -out server.csr \
  -subj "/CN=TLS-Theater-Server"

完了したら、server.csrを認証局に送信、署名してもらおう！

In [ ]:
!cat server.csr

## 0.3 サーバー証明書のインポート

In [ ]:
# @title {"run":"auto"}
# @markdown CAから受け取ったキミのサーバー証明書を貼り付けて!
server_crt = "-----BEGIN CERTIFICATE----- … -----END CERTIFICATE-----" # @param {"type":"string","placeholder":"CA証明書の中身を貼ってください"}

# crt形式に成形
server_crt = server_crt.replace(" ", "\n")
server_crt = server_crt.replace("-----BEGIN\nCERTIFICATE-----", "-----BEGIN CERTIFICATE-----")
server_crt = server_crt.replace("-----END\nCERTIFICATE-----", "-----END CERTIFICATE-----")
open("server.crt", "w").write(server_crt)
print(server_crt)
print("アップロード完了")

-----BEGIN CERTIFICATE-----
…
-----END CERTIFICATE-----
アップロード完了


# I. ハンドシェイクプロトコル

## 1. まずは挨拶

### 1.1 クライアントハローを受け取ろう！

クライアントから
- 暗号スイート
- クライアントランダム
が送られてくるよ！

実際のTLSではもう少したくさんのパラメータが渡されるよ

In [ ]:
# @title {"run":"auto"}
# @markdown クライアントハローを受け取ったらクライアントランダムを保存よう

client_random = "" # @param {"type":"string","placeholder":"クライアントランダム"}


### 1.2 サーバーハローを返そう


#### 1. サーバーランダム作り
下のセルで
```
!openssl rand -hex 32
```
を実行し、サーバーランダムを生成できるよ

In [ ]:
!openssl rand -hex 32

In [ ]:
# @title {"run":"auto"}
server_random = "" # @param {"type":"string","placeholder":"生成できたサーバーランダム"}
print(server_random)

### 1.3 サーバーハローを送ろう



- 使用するバージョン
- 現在時刻
- サーバーランダム
- セッションID
- 使用する暗号スイート
- 使用する圧縮方法

を返すけど、今回は
- 暗号スイート: `TLS_RSA_WITH_AES_128_CBC_SHA256`
- サーバーランダム

だけでいいよ(簡略化)

ついでに`TLS_RSA_WITH_AES_128_CBC_SHA256`の意味は
```
TLS _ RSA _ WITH _ AES_128_CBC _ SHA256
 |     |            |               |
 |     |            |               └ MAC に使うハッシュ: HMAC-SHA256
 |     |            └ 共通鍵暗号: AES 128bit CBC モード
 |     └ 鍵交換: RSA
 └ プロトコル
 ```

 だよ！





### 1.4 証明書を渡そう

さっきCAから貰った証明書をクライアントに送ろう!

証明書は
```
!cat server.crt
```
で確認できるよ

また、フィンガープリントは
```
!openssl x509 -in server.crt -fingerprint -sha256 -noout
```
だ！

### 1.5 サーバーハローを終わらせよう

サーバーハローは終わりだって伝えよう



## 2. ClientKeyExchange-共通の秘密情報を作る

### 2.1 プレマスターシークレットを受け取ろう

base64でエンコードされたプレマスターシークレットを受け取れたかな？

### 2.2 プレマスターシークレットを復号しよう

プレマスターシークレットは暗号化されている。
サーバーの秘密鍵で復号しよう。
```
! touch pms.bin
!echo "プレマスターシークレットのbase64" | base64 -d |\
  openssl pkeyutl -decrypt \
  -inkey server.key \
  -out   pms.bin
```
`pms.bin`が復号されたプリマスターシークレットだ!



### 2.3 共通のマスターシークレットを作ろう



プレマスターシークレットを16進数に変換するよ!

`!od -An -tx1 pms.bin | tr -d ' \n'`.  
ここではObjectDumpコマンドで`pms.bin`を16進数(TypeHex)にして、その出力から改行を取り除いてるんだ。


In [ ]:
!od -An -tx1 pms.bin | tr -d ' \n'

In [ ]:
pms = "" # @param {"type":"string","placeholder":"上で出力されたpms"}


In [ ]:
# @markdown 次はサーバーランダムとクライアントランダムを使うよ。

# @markdown 上にさかのぼるのはめんどくさいから、このセルを実行すれば再確認できるようにしといたよ

print(f"client_random     = {client_random}")
print(f"server_random     = {server_random}")

ここでは3つの素材(プレマスターシークレット、サーバーランダム、クライアントランダム)から7つの鍵を作るよ。

- master_secret: 48byte
- client_write_MAC: 32byte
- server_write_MAC: 32byte
- client_write_key: 16byte
- server_write_key: 16byte
- client_write_iv: 16byte
- server_write_iv: 16byte

今回は簡略化のため、シンプルな方法をとるよ。

```python
#hashlib.sha256がstr受け付けないからbyteに変換するよ
base = bytes.fromhex(pms + client_random + server_random)
# sha256の出力が32byteだから2回やる
h1 = hashlib.sha256(base + b"ms1").digest() # bをつけるとbyte列として解釈されるよ
h2 = hashlib.sha256(base + b"ms2").digest()
master_secret = (h1 + h2)[:48].hex() # 最後に16進数文字列に変換
```
これを参考に残りの鍵を作ってみよう!

In [ ]:

import hashlib
#hashlib.sha256がstr受け付けないからbyteに変換するよ
base = bytes.fromhex(pms + client_random + server_random)
# sha256の出力が32byteだから2回やる
h1 = hashlib.sha256(base + b"ms1").digest() # bをつけるとbyte列として解釈されるよ
h2 = hashlib.sha256(base + b"ms2").digest()
master_secret = (h1 + h2)[:48].hex() # 最後に16進数文字列に変換

# 今度は master_secret を base にする
ms_base =  bytes.fromhex(master_secret + client_random + server_random)
# ここに追記
client_write_MAC = hashlib.sha256(ms_base + b"client_write_MAC").digest()[:32].hex()
#server_write_MAC
#client_write_key
#server_write_key
#client_write_iv
#server_write_iv

print(f"master_secret    = {master_secret}")
print(f"client_write_MAC = {client_write_MAC}")
#print(f"server_write_MAC = {server_write_MAC}")
#print(f"client_write_key = {client_write_key}")
#print(f"server_write_key = {server_write_key}")
#print(f"client_write_iv  = {client_write_iv}")
#print(f"server_write_iv  = {server_write_iv}")
print()
print("✓ 鍵導出完了！この値は絶対に Discord に貼らないこと 🔒")


master_secret    = 1328cbfaeb685fd0c438500350904a06ec81a9148c3d6a13bdb5dc5ca18b584b62fba8886a3901c22c9e94624c0fba22
client_write_MAC = b493e7bf81608b9cada808544dba2e30c58358aee182fe0090c6f0457422ea1c
server_write_MAC = 812dac9858d182fc9ea7d2b13b14efeed7f2aee87597812066f3308178f79a64
client_write_key = ed56337ad9bbc14996b84e272cf7ae63
server_write_key = e24872082d7afa8c5f95f3425a70beff
client_write_iv  = cd80ddf644f224ff952870ef9a52706f
server_write_iv  = 1aaf4170a4cb852d9bad9d06561d42bd

✓ 鍵導出完了！この値は絶対に Discord に貼らないこと 🔒


ついでに: Colabの挿入>スクラッチセルを選択し、
```python
print(f"master_secret    = {master_secret}")
print(f"client_write_MAC = {client_write_MAC}")
print(f"server_write_MAC = {server_write_MAC}")
print(f"client_write_key = {client_write_key}")
print(f"server_write_key = {server_write_key}")
print(f"client_write_iv  = {client_write_iv}")
print(f"server_write_iv  = {server_write_iv}")
```
を実行しておくと、この後便利だぞ！

## 3. 暗号文でのやりとりする準備をしよう！

### 3.1 クライアントのハンドシェイクプロトコル終了を待とう
しばらくしたら、クライアントから暗号文と、ハンドシェイク終了の合図が来るよ

その間に、今までの使っていたDiscordチャンネルのメッセージをコピペして、ハッシュを求めよう!

ハッシュを求めるのは
```shell
!openssl dgst -sha256 -hex （ハッシュを求めたいファイル） | cut -d' ' -f2
```
だよ！

In [ ]:
# @title ログをファイルに保存する
chat_log = "" # @param {"type":"string","placeholder":"ハンドシェイクチャンネルのログを貼ろう!"}
open("hs_log.txt", "w").write(chat_log)
print("hs_log.txtに保存")

hs_log.txtに保存


In [ ]:
# @title hs_hash を保存する {"run":"auto"}
hs_hash = "" # @param {"type":"string","placeholder":"openssl dgst の出力（64文字のhex）"}
print(f"hs_hash = {hs_hash}")

hs_hash = 


### 3.2 クライアントの暗号文を解読する

以下でクライアントの暗号文を解読できるよ！

```shell
!echo "（クライアントの base64）" | base64 -d | \
  openssl enc -d -aes-128-cbc \
  -K "（client_write_key）" -iv "（client_write_iv）" -nosalt
```

復号結果はさっき求めたハッシュと同じになったかな？

### 3.3. サーバーの Finished を送ろう

今度はサーバー側の `hs_hash`を `server_write_key` で暗号化して返すよ。


```shell
!echo -n "（さっき求めたhs_hash）" | \
  openssl enc -aes-128-cbc \
  -K "（server_write_key）" -iv "（server_write_iv）" -nosalt | base64 -w 0
```

出力されたbase64をDiscordで送ろう!

例：
```
ChangeCipherSpec です。
Finished（暗号文）:
（base64 の出力をそのまま貼る）

ハンドシェイク以上です。
```

# II. 暗号通信開始！アプリケーションデータプロトコル

アプリケーションデータプロトコルのデータは、【平文 + MAC値】を暗号化したものだよ。

MAC値は暗号学入門p.207でやったね。

これを使うと、メッセージが改ざんされたか調べることができるよ。

### 1.1 暗号文を復号する

まずは、送られてきた暗号文を復号してみよう！
```shell
!echo "（クライアントの base64）" | base64 -d | \
  openssl enc -d -aes-128-cbc \
  -K "（client_write_key）" -iv "（client_write_iv）" -nosalt
  ```

```
平文:
MAC値:
```
の形式になったかな？

### 1.2 メッセージの検証

MAC値は『"メッセージのハッシュ値"を共通のMAC鍵とハッシュ化したもの』だね。

今回のメッセージは『平文』だよ。

MAC値を求める方法はこうだよ。MAC値が同じか検証してみよう！
```shell
# MAC用のオプションをつけてハッシュ化
!echo -n "（平文）" | openssl dgst -sha256 -mac HMAC \
  -macopt hexkey:"（client_write_MAC）" -hex | cut -d' ' -f2
```


同じMAC値になったかな？

## 2. 暗号文を送ろう！

今度は暗号文を送ってみよう。
必要なのは

- 送りたいメッセージ
- MAC値

の2つだ！さっき受け取ったのと同じだね。

### 2.1 送る文を考えよう！

まずは『送りたいメッセージ』を決めよう！  
明日の予定でも秘密の約束でも、愛の告白でもなんでもいいよ！

### 2.2 MAC値を求めよう！

決めたらMAC値を求めよう  
方法はもう知ってるよね？

1つ注意：今回はサーバーが書き込むから使う鍵が変わるよ！



### 2.3 暗号化しよう！

```shell
!echo -e "平文:（さっき考えた文）\nMAC値:（求めたMAC値）" | \
  openssl enc -aes-128-cbc \
  -K "（server_write_key）" -iv "（server_write_iv）" -nosalt | base64 -w 0
```

無事暗号化できたかな？

クライアントが受け取って内容を読めたら、TLSでの通信完了だよ！

# III. 完了！

おつかれさま！！

パケットはみんなに見えていたのに、内容はバレなかったよね。

これがTLSだよ！